# 🤗 Hugging Face — Publicando seu Modelo no Hub
**FIAP Postech — IA para Devs | Fase 1 · Fundamentos de IA e Machine Learning**

---

## O que você vai aprender
1. O que é o Hugging Face Hub e por que usá-lo
2. Criar e configurar sua conta + token de acesso
3. Salvar um modelo sklearn no formato correto
4. Criar o `model card` (README) — obrigatório para boas práticas
5. Fazer upload do modelo para o Hub
6. Carregar e usar o modelo publicado (como qualquer pessoa faria)
7. Publicar um pipeline sklearn como módulo reutilizável

---
> **Por que o Hugging Face?**  
> É o GitHub dos modelos de ML. Permite versioning, colaboração, e qualquer pessoa do mundo pode baixar e usar seu modelo com 2 linhas de código.

---
## Contexto: Ciclo de Vida de um Modelo ML em Produção

Antes de publicar, é fundamental entender **onde o deploy se encaixa** no processo de Machine Learning. Qualquer projeto passa pelas seguintes fases:

| Fase | Descrição | Responsável Típico |
|---|---|---|
| **Coleta de Dados** | ETL de múltiplas fontes — Data Warehouses, Data Lakes, APIs | Data Engineer |
| **Pré-processamento** | Limpeza, transformação, feature engineering | Data Scientist |
| **Treinamento** | Seleção de algoritmo, ajuste de hiperparâmetros | Data Scientist |
| **Validação** | Avaliação em dados de teste, comparação de versões | Data Scientist |
| **Deploy (Implantação)** | Empacotamento, versionamento e exposição como serviço | ML Engineer |
| **Monitoramento** | Data drift, concept drift, degradação de performance | ML Engineer / DevOps |

### O gargalo real: Validação → Produção

Rastrear cada etapa manualmente para criar a versão final do modelo é um processo trabalhoso e propenso a erros. **É aqui que o Hugging Face Hub resolve o problema** — ele padroniza o empacotamento e o versionamento, encurtando drasticamente esse gap.

### Por que versionar modelos?

Assim como em software tradicional, é essencial:
- Fazer **rollback** para versão anterior se um novo modelo degradar em produção
- **Rastrear** qual dado + código gerou qual modelo (reprodutibilidade)
- **Colaborar** sem sobrescrever o trabalho de outras pessoas da equipe

```
Sem versionamento:  modelo_final_v3_FINAL_revisado_OK.pkl  ← caos
Com Hugging Face:   usuario/meu-modelo  @ commit abc123f   ← profissional
```

---
## O Ecossistema Hugging Face

O Hugging Face começou como uma startup de chatbots em 2016 e se tornou o **"GitHub dos modelos de ML"** — a maior plataforma colaborativa de IA do mundo, com mais de **1 milhão de modelos públicos**.

### Componentes Principais

| Componente | O que é | Para que serve |
|---|---|---|
| **🤗 Hub** | Repositório central | Versionar e compartilhar modelos, datasets e Spaces |
| **Transformers** | Biblioteca Python | +300 mil modelos pré-treinados para NLP, visão, áudio |
| **Datasets** | Biblioteca Python | +100 mil datasets com streaming e cache eficiente |
| **Spaces** | Plataforma de deploy | Hospedagem gratuita de demos Gradio/Streamlit |
| **Inference API** | REST API | Usar qualquer modelo via HTTP sem infraestrutura própria |
| **PEFT / TRL** | Bibliotecas Python | Fine-tuning eficiente (LoRA, QLoRA) e RLHF |

### Por que profissionais de TI precisam saber isso?

Mesmo quem não é Cientista de Dados interage com o Hub: a tarefa de levar um modelo a produção — empacotamento, versionamento e exposição como serviço — é responsabilidade do **Engenheiro de ML** ou do **Dev Backend**. O Hub padroniza esse processo.

> **Analogia:** Hugging Face está para modelos de ML assim como o npm está para pacotes JavaScript ou o PyPI para pacotes Python.

## 0. Instalação

> Instale o núcleo obrigatório. As opcionais (`transformers`, `gradio`) só são necessárias nas seções 9 e 10.

In [ ]:
# Obrigatórias
!pip install huggingface_hub scikit-learn joblib --quiet

# Opcionais — descomente conforme avançar nas seções
# !pip install transformers torch --quiet   # Seção 9
# !pip install gradio --quiet               # Seção 10

print('✅ Dependências instaladas')

In [ ]:
import numpy as np
import pandas as pd
import joblib
import json
import os
from pathlib import Path

from sklearn.datasets import load_breast_cancer
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score

from huggingface_hub import HfApi, HfFolder, login, Repository

print('✅ Imports OK')

---
## 1. Configuração do Hugging Face

### Criando sua conta e token de acesso

1. Acesse **huggingface.co** e crie uma conta gratuita
2. Clique na sua foto → **Settings → Access Tokens → New Token**
3. Escolha o tipo **`write`** (obrigatório para fazer upload)
4. Copie o token e cole abaixo

> ⚠️ **NUNCA** faça commit do token no Git. Use variável de ambiente ou `.env`.

---

### Autenticação via CLI (alternativa ao código Python)

Se preferir autenticar pelo terminal antes de abrir o notebook:

```bash
huggingface-cli login
# → Pedirá o token de forma segura; armazena em ~/.cache/huggingface/token
```

Isso é especialmente útil em ambientes de servidor ou CI/CD.

In [ ]:
# Opção 1: Login interativo (recomendado — pede o token de forma segura)
login()  # vai pedir o token via input seguro

# Opção 2: Via variável de ambiente (para uso em CI/CD)
# import os
# login(token=os.environ['HF_TOKEN'])

---
## 2. Treinando o Modelo

In [ ]:
# Treinando o pipeline completo
data = load_breast_cancer()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = data.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('classifier', RandomForestClassifier(n_estimators=100, random_state=42))
])

pipeline.fit(X_train, y_train)
y_pred = pipeline.predict(X_test)
y_proba = pipeline.predict_proba(X_test)[:, 1]

auc = roc_auc_score(y_test, y_proba)
print('📊 Métricas do modelo:')
print(classification_report(y_test, y_pred, target_names=data.target_names))
print(f'AUC Score: {auc:.4f}')

---
## 3. Preparando os Arquivos para Upload

O Hugging Face Hub espera uma estrutura de repositório. Vamos criar:

In [ ]:
# ⚠️ ALTERE PARA O SEU USUÁRIO DO HUGGING FACE
HF_USERNAME = 'seu-usuario-aqui'  # ex: 'joaosilva'
REPO_NAME = 'breast-cancer-classifier-fiap'
REPO_ID = f'{HF_USERNAME}/{REPO_NAME}'

# Criando diretório local do modelo
model_dir = Path('./modelo_hf')
model_dir.mkdir(exist_ok=True)
print(f'📁 Diretório: {model_dir}')

In [ ]:
# 3.1 Salvando o pipeline sklearn com joblib
model_path = model_dir / 'model.joblib'
joblib.dump(pipeline, model_path)
print(f'✅ Modelo salvo: {model_path} ({model_path.stat().st_size / 1024:.1f} KB)')

In [ ]:
# 3.2 Salvando metadados do modelo (config.json)
from sklearn.metrics import recall_score, f1_score, accuracy_score

config = {
    'model_type': 'sklearn_pipeline',
    'algorithm': 'RandomForestClassifier',
    'task': 'binary_classification',
    'dataset': 'Wisconsin Breast Cancer (sklearn)',
    'n_features': len(data.feature_names),
    'feature_names': list(data.feature_names),
    'classes': list(data.target_names),
    'hyperparameters': {
        'n_estimators': 100,
        'random_state': 42
    },
    'metrics': {
        'accuracy': accuracy_score(y_test, y_pred),
        'recall': recall_score(y_test, y_pred),
        'f1_score': f1_score(y_test, y_pred),
        'auc_roc': auc
    },
    'train_size': len(X_train),
    'test_size': len(X_test),
    'sklearn_version': __import__('sklearn').__version__
}

config_path = model_dir / 'config.json'
with open(config_path, 'w') as f:
    json.dump(config, f, indent=2)

print('✅ config.json salvo:')
print(json.dumps(config, indent=2))

In [ ]:
# 3.3 Criando o Model Card (README.md)
# O model card é a "documentação" do seu modelo — OBRIGATÓRIO para boas práticas

model_card = f"""---
license: mit
tags:
- sklearn
- classification
- healthcare
- breast-cancer
- random-forest
metrics:
- accuracy
- f1
- roc_auc
language:
- pt
---

# 🩺 Breast Cancer Classifier — FIAP Postech IA para Devs

Modelo de classificação binária para predição de tumores malignos/benignos 
treinado no dataset Wisconsin Breast Cancer.

Desenvolvido como projeto acadêmico para a Pós-Graduação **IA para Devs** da FIAP Postech.

## Sobre o Modelo

| Atributo | Valor |
|---|---|
| Algoritmo | Random Forest Classifier |
| Pré-processamento | StandardScaler |
| Dataset | Wisconsin Breast Cancer (sklearn) |
| Features | 30 features numéricas |
| Classes | malignant (0), benign (1) |

## Métricas de Desempenho

| Métrica | Valor |
|---|---|
| Accuracy | {config['metrics']['accuracy']:.4f} |
| Recall | {config['metrics']['recall']:.4f} |
| F1 Score | {config['metrics']['f1_score']:.4f} |
| AUC-ROC | {config['metrics']['auc_roc']:.4f} |

> **Métrica principal: Recall** — em diagnóstico oncológico, minimizamos falsos negativos.

## Como Usar

```python
from huggingface_hub import hf_hub_download
import joblib

# Download do modelo
model_path = hf_hub_download(
    repo_id='{REPO_ID}',
    filename='model.joblib'
)
model = joblib.load(model_path)

# Predição
# X deve ter as 30 features do dataset Wisconsin Breast Cancer
predicao = model.predict(X_novo)
probabilidade = model.predict_proba(X_novo)[:, 1]
```

## Features Esperadas

O modelo espera as 30 features numéricas do Wisconsin Breast Cancer:
{', '.join(data.feature_names)}

## Limitações e Avisos Éticos

⚠️ Este modelo é **apenas para fins acadêmicos**. Não deve ser usado para diagnóstico clínico real.

## Contato

Desenvolvido por: {HF_USERNAME}  
Curso: FIAP Postech — IA para Devs  
"""

readme_path = model_dir / 'README.md'
with open(readme_path, 'w', encoding='utf-8') as f:
    f.write(model_card)

print('✅ README.md (Model Card) salvo')
print('\n📁 Arquivos prontos para upload:')
for f in model_dir.iterdir():
    print(f'   {f.name} ({f.stat().st_size / 1024:.1f} KB)')

---
## 4. Upload para o Hugging Face Hub

In [ ]:
api = HfApi()

# Criando o repositório no Hub (se não existir)
try:
    api.create_repo(
        repo_id=REPO_ID,
        repo_type='model',
        private=False,  # True para privado
        exist_ok=True   # não dá erro se já existir
    )
    print(f'✅ Repositório criado/verificado: https://huggingface.co/{REPO_ID}')
except Exception as e:
    print(f'❌ Erro ao criar repositório: {e}')

In [ ]:
# Upload de todos os arquivos
arquivos = ['model.joblib', 'config.json', 'README.md']

for arquivo in arquivos:
    local_path = model_dir / arquivo
    try:
        api.upload_file(
            path_or_fileobj=str(local_path),
            path_in_repo=arquivo,
            repo_id=REPO_ID,
            repo_type='model',
            commit_message=f'Add {arquivo}'
        )
        print(f'✅ {arquivo} enviado')
    except Exception as e:
        print(f'❌ Erro no upload de {arquivo}: {e}')

print(f'\n🎉 Modelo publicado em: https://huggingface.co/{REPO_ID}')

---
## 5. Carregando o Modelo Publicado
Simulando como qualquer pessoa faria para usar seu modelo.

In [ ]:
from huggingface_hub import hf_hub_download

# Download do modelo do Hub
downloaded_model_path = hf_hub_download(
    repo_id=REPO_ID,
    filename='model.joblib'
)

# Carregando
modelo_carregado = joblib.load(downloaded_model_path)
print(f'✅ Modelo carregado de: {REPO_ID}')
print(f'   Tipo: {type(modelo_carregado)}')
print(f'   Steps: {list(modelo_carregado.named_steps.keys())}')

In [ ]:
# Testando o modelo carregado
y_pred_hf = modelo_carregado.predict(X_test)
y_proba_hf = modelo_carregado.predict_proba(X_test)[:, 1]

print('📊 Predição com modelo baixado do Hub:')
print(classification_report(y_test, y_pred_hf, target_names=data.target_names))
print(f'AUC: {roc_auc_score(y_test, y_proba_hf):.4f}')

# Predição em uma amostra nova
amostra = X_test.iloc[[0]]
pred = modelo_carregado.predict(amostra)[0]
prob = modelo_carregado.predict_proba(amostra)[0]
print(f'\n🔍 Exemplo de predição:')
print(f'   Predição: {data.target_names[pred]} ({["malignant", "benign"][pred]})')
print(f'   Probabilidade: malignant={prob[0]:.3f}, benign={prob[1]:.3f}')

---
## 6. Carregando os Metadados

In [ ]:
config_path_hf = hf_hub_download(repo_id=REPO_ID, filename='config.json')

with open(config_path_hf) as f:
    config_carregado = json.load(f)

print('📋 Metadados do modelo:')
print(f'   Algoritmo: {config_carregado["algorithm"]}')
print(f'   Features: {config_carregado["n_features"]}')
print(f'   Métricas registradas:')
for k, v in config_carregado['metrics'].items():
    print(f'     {k}: {v:.4f}')

---
## 7. Extras — Boas Práticas

### Versionamento
Cada upload cria um commit no repositório. Para versionar explicitamente:

```python
api.upload_file(
    ...,
    commit_message='v1.1 — adicionado Gradient Boosting, recall melhorou 2%'
)
```

### Buscar modelos do Hub (uso avançado)

```python
from huggingface_hub import list_models

# Listar modelos de classificação com sklearn
modelos = list(list_models(filter='sklearn', task='tabular-classification'))
for m in modelos[:5]:
    print(m.id, m.downloads)
```

### Dataset no Hub

```python
api.create_repo(repo_id='meu-usuario/meu-dataset', repo_type='dataset')
api.upload_file(path_or_fileobj='data.csv', path_in_repo='data.csv',
                repo_id='meu-usuario/meu-dataset', repo_type='dataset')
```

---
## Resumo do Fluxo

```
Treinar modelo → Salvar com joblib → Criar config.json + README.md
     ↓
api.create_repo() → api.upload_file() × 3
     ↓
https://huggingface.co/seu-usuario/seu-modelo
     ↓
Qualquer pessoa: hf_hub_download() → joblib.load() → .predict()
```

---
## 8. Deploy com Gradio Spaces — Interface Interativa

Publicar o modelo no Hub resolve o problema de distribuição. Mas para stakeholders não-técnicos, o que convence é uma **interface onde eles podem testar ao vivo**.

O Hugging Face **Spaces** resolve isso: hospedagem gratuita de demos Python, sem DevOps.

### O que é um Space?

Um repositório especial no Hub que executa um `app.py` (Gradio ou Streamlit) em containers gerenciados pelo Hugging Face.

```
Escrever app.py → Criar Space no Hub → Upload dos arquivos → URL pública em minutos
```

### Tiers disponíveis (2025)

| Tier | Hardware | Uso ideal |
|---|---|---|
| **Free** | 2 vCPU, 16 GB RAM | Sklearn, NLP leve, demos acadêmicas |
| **Small** | 2 vCPU, 16 GB + persistência | APIs com estado |
| **GPU T4** | NVIDIA T4 | Modelos de linguagem médios (7B) |
| **GPU A10G** | NVIDIA A10G | Modelos grandes, difusão de imagens |

### Por que Gradio?

- **4 linhas de código** para criar um formulário web funcional
- Suporta sliders, imagens, áudio, vídeo, dataframes como inputs/outputs
- Gera automaticamente uma REST API além da UI
- Integra com o Hub: `gr.load('usuario/modelo')` serve qualquer modelo do Hub

In [ ]:
# ============================================================
# Gera o arquivo app.py para publicar no Hugging Face Spaces
# ============================================================

gradio_app_code = f'''import gradio as gr
import joblib
import numpy as np
import json
from huggingface_hub import hf_hub_download

# Carrega modelo e metadata direto do Hub
model_path = hf_hub_download(repo_id="{REPO_ID}", filename="model.joblib")
config_path = hf_hub_download(repo_id="{REPO_ID}", filename="config.json")

model = joblib.load(model_path)
with open(config_path) as f:
    config = json.load(f)

FEATURES = config["feature_names"][:10]  # primeiras 10 para a demo

def predict(*values):
    """Recebe os sliders e retorna o diagnóstico com probabilidades."""
    X_full = np.zeros((1, config["n_features"]))
    X_full[0, :len(values)] = values
    pred = model.predict(X_full)[0]
    prob = model.predict_proba(X_full)[0]
    return {{
        config["classes"][0]: float(prob[0]),
        config["classes"][1]: float(prob[1])
    }}

inputs = [gr.Slider(minimum=0, maximum=30, value=14, label=f) for f in FEATURES]

demo = gr.Interface(
    fn=predict,
    inputs=inputs,
    outputs=gr.Label(num_top_classes=2, label="Diagnóstico"),
    title="🩺 Classificador de Câncer de Mama",
    description=(
        "Demo acadêmica — **NÃO utilizar para diagnóstico clínico real.**\\n"
        f"Modelo: RandomForest | AUC: {{config[\'metrics\'][\'auc_roc\']:.4f}}"
    ),
    examples=[[14]*10],
    theme=gr.themes.Soft()
)

if __name__ == "__main__":
    demo.launch()
'''

# Salva o app.py localmente
with open('app.py', 'w', encoding='utf-8') as f:
    f.write(gradio_app_code)

print("✅ app.py criado!")
print()
print("Para publicar o Space:")
print(f"  api.create_repo(repo_id='{HF_USERNAME}/breast-cancer-demo', repo_type='space', space_sdk='gradio')")
print(f"  api.upload_file(path_or_fileobj='app.py', path_in_repo='app.py',")
print(f"                  repo_id='{HF_USERNAME}/breast-cancer-demo', repo_type='space')")
print()
print(f"  → https://huggingface.co/spaces/{HF_USERNAME}/breast-cancer-demo")

---
## 9. Usando Modelos Pré-Treinados com `transformers`

O grande diferencial do Hugging Face é o acesso a **+1 milhão de modelos pré-treinados** via a biblioteca `transformers` — tudo open source.

### O que são modelos pré-treinados?

São modelos treinados em enormes volumes de dados por organizações como Google, Meta e Hugging Face. Você pode:
- **Usar diretamente** via `pipeline()` — zero treino necessário
- **Fazer fine-tuning** — adaptar ao seu domínio com poucos dados e custo baixo

### Por que isso importa?

Treinar um BERT do zero custaria **semanas de GPU e dezenas de milhares de dólares**. Com modelos pré-treinados, você usa esse conhecimento como ponto de partida:

```
Dado Novo → Modelo Pré-Treinado (conhecimento geral)
                        ↓  fine-tuning (seus dados, horas de treino)
              Modelo Especializado (seu domínio, seu idioma)
```

### Tarefas disponíveis via `pipeline()`

| Task | Exemplo de uso |
|---|---|
| `sentiment-analysis` | Monitoramento de redes sociais, reviews de produtos |
| `zero-shot-classification` | Classificar textos sem rótulos de treino |
| `ner` (extração de entidades) | Reconhecer nomes, datas, organizações em contratos |
| `text-generation` | Completar código, gerar rascunhos |
| `translation` | Tradução automática (100+ idiomas) |
| `question-answering` | Chatbots sobre documentos internos |
| `image-classification` | Triagem de imagens médicas, qualidade industrial |

> 💡 A `pipeline()` funciona para NLP, visão computacional e áudio — mesma API.

In [ ]:
# ============================================================
# Modelos pré-treinados com transformers.pipeline()
# Requer: pip install transformers torch
# ============================================================
# Descomente o bloco que quiser testar

# ------ 1. Análise de sentimento (multilingual) ------
# from transformers import pipeline
# sentiment = pipeline(
#     'sentiment-analysis',
#     model='lxyuan/distilbert-base-multilingual-cased-sentiments-student'
# )
# print(sentiment("Este modelo é fantástico para produção!"))
# print(sentiment("O treinamento demorou muito e os resultados foram ruins."))

# ------ 2. Classificação zero-shot (sem treino extra!) ------
# classifier = pipeline('zero-shot-classification', model='facebook/bart-large-mnli')
# resultado = classifier(
#     "O paciente apresenta nódulo suspeito no pulmão direito",
#     candidate_labels=["oncologia", "cardiologia", "neurologia", "ortopedia"]
# )
# for label, score in zip(resultado['labels'], resultado['scores']):
#     print(f"  {label}: {score:.1%}")

# ------ 3. Geração de texto em português ------
# gerador = pipeline('text-generation', model='pierreguillou/gpt2-small-portuguese')
# print(gerador("A inteligência artificial vai", max_length=60, num_return_sequences=2))

# ------ 4. Extração de informação (NER) ------
# ner = pipeline('ner', model='lener-br/lener-br', aggregation_strategy='simple')
# print(ner("O juiz João da Silva condenou a empresa Petrobras em São Paulo"))

# ------ 5. Similaridade semântica ------
# from transformers import AutoTokenizer, AutoModel
# import torch
# # Útil para busca semântica e sistemas de recomendação

print("💡 Descomente os blocos acima para experimentar.")
print("   Cada pipeline() baixa o modelo automaticamente na primeira execução.")
print("   Os pesos ficam em cache em ~/.cache/huggingface/hub/")

---
## 10. Boas Práticas de MLOps com Hugging Face

MLOps é a disciplina que une Machine Learning e DevOps para levar modelos a produção de forma confiável, rastreável e reprodutível.

### Versionamento semântico

```python
# Cada upload cria um commit no histórico do repositório
api.upload_file(
    path_or_fileobj='model.joblib',
    path_in_repo='model.joblib',
    repo_id=REPO_ID,
    commit_message='feat: v2.0 — substituiu RF por XGBoost, AUC +3%'
)

# Criar uma tag de versão (como no Git)
api.create_tag(repo_id=REPO_ID, tag='v1.0.0', tag_message='Versão estável inicial')

# Listar histórico de commits
commits = list(api.list_repo_commits(repo_id=REPO_ID))
for c in commits[:3]:
    print(c.commit_id[:7], '|', c.title)
```

### Controle de visibilidade

```python
# Manter privado durante desenvolvimento, publicar ao validar
api.update_repo_visibility(repo_id=REPO_ID, private=True)   # dev
api.update_repo_visibility(repo_id=REPO_ID, private=False)  # produção
```

### Monitoramento de uso

```python
# Ver estatísticas de download do seu modelo
info = api.model_info(repo_id=REPO_ID)
print(f'Downloads último mês: {info.downloads}')
print(f'Likes: {info.likes}')
```

### Checklist antes de publicar

- [ ] Model card completo — métricas, dados de treino, limitações
- [ ] Licença definida (`mit`, `apache-2.0`, `cc-by-4.0`...)
- [ ] Aviso ético se o modelo puder causar viés ou dano
- [ ] Exemplo funcional no README com `pip install` e código pronto para copiar
- [ ] `config.json` com hiperparâmetros **e** versões de todas as dependências
- [ ] Tags corretas para aparecer nas buscas do Hub (`sklearn`, `classification`, idioma...)

### Quando usar cada estratégia de deploy

| Cenário | Solução HF | Custo |
|---|---|---|
| Demo rápida para stakeholders | Gradio Spaces (free tier) | Gratuito |
| API de inferência para protótipo | Inference API (hosted) | Gratuito (com limites) |
| Produção com SLA | Inference Endpoints | Pago (por hora de GPU/CPU) |
| Modelo enorme (LLM) | Inference Endpoints + quantização | Pago |

---
## Resumo e Próximos Passos

### O que você aprendeu nesta aula

| # | Habilidade |
|---|---|
| ✅ | Entender o ciclo de vida completo de ML em produção |
| ✅ | Criar conta e token de acesso no Hugging Face |
| ✅ | Publicar modelos sklearn com joblib + metadata |
| ✅ | Escrever Model Cards profissionais |
| ✅ | Fazer upload e download via `HfApi` |
| ✅ | Criar demos interativas com Gradio Spaces |
| ✅ | Usar modelos pré-treinados com `transformers.pipeline()` |
| ✅ | Aplicar boas práticas de MLOps e versionamento |

### Próximos passos recomendados

1. **Fine-tuning** — ajustar BERT ou GPT-2 para seu domínio específico com `Trainer` do HF
2. **Inference Endpoints** — deploy serverless de modelos maiores diretamente no Hub
3. **Quantização** — reduzir tamanho de modelos com GGUF/GPTQ para rodar localmente
4. **Avaliação rigorosa** — biblioteca `evaluate` do Hugging Face para métricas padronizadas
5. **MLflow / Weights & Biases** — rastreamento de experimentos antes do deploy

---

> **Desafio prático:** publique um modelo treinado com seus próprios dados no Hub, crie um Space com Gradio e compartilhe o link com a turma!

---
*FIAP Postech — IA para Devs | Aula 06 — Publicação de modelos no Hugging Face*